# 🤖 Models Cheatsheet — DSI SquarePoint

Полный разбор каждой модели: теория, гиперпараметры, когда использовать, как диагностировать.  
В конце — copy-paste шаблон pipeline для любого датасета.

---

## Содержание

1. [Baseline — DummyRegressor](#1)
2. [Ridge Regression](#2)
3. [Lasso & ElasticNet](#3)
4. [Logistic Regression (классификация)](#4)
5. [LightGBM](#5)
6. [XGBoost](#6)
7. [Сравнение моделей — когда что выбрать](#7)
8. [Cross-Validation стратегии](#8)
9. [Overfitting — диагностика и лечение](#9)
10. [🚀 Copy-paste шаблон pipeline](#10)


## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import (
    train_test_split, KFold, cross_val_score,
    TimeSeriesSplit, learning_curve
)
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, roc_auc_score, classification_report
)
from sklearn.linear_model import (
    Ridge, RidgeCV, Lasso, LassoCV,
    ElasticNet, ElasticNetCV, LogisticRegression
)
from sklearn.dummy import DummyRegressor, DummyClassifier
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
pd.set_option('display.float_format', '{:.4f}'.format)

SEED   = 42
TARGET = 'target'
np.random.seed(SEED)

# ── Синтетический датасет (регрессия) ─────────────────────────────────
n = 1000
X_raw = pd.DataFrame({
    'glucose':    np.random.normal(100, 20, n),
    'bmi':        np.random.normal(26, 5, n),
    'age':        np.random.randint(20, 80, n).astype(float),
    'hba1c':      np.random.normal(5.5, 1.2, n),
    'revenue':    np.random.exponential(1000, n),
    'segment':    np.random.choice(['A','B','C'], n),
})
y_raw = (
    0.5 * X_raw['glucose'] +
    0.3 * X_raw['bmi'] +
    0.2 * X_raw['hba1c'] * 10 +
    np.random.normal(0, 5, n)
)
print('Dataset ready:', X_raw.shape)

---
## 1. Baseline — DummyRegressor / DummyClassifier

### Зачем

Нижняя планка. Любая модель которая не бьёт baseline — бесполезна.  
**На DSI: всегда первый шаг**, прежде чем обучать что-то умное.

### Что делает

- `strategy='mean'` → всегда предсказывает среднее тренировочного таргета  
- `strategy='median'` → предсказывает медиану (устойчивее при выбросах)  
- `strategy='constant'` → предсказывает заданное значение

### Метрики baseline

- **RMSE baseline = std(y_train)** — именно столько ошибки у mean-предсказателя  
- **R² baseline = 0.00** по определению (вся дисперсия необъяснена)  
- **Spearman ρ baseline ≈ 0** — не ранжирует вообще


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_raw.select_dtypes(include=np.number), y_raw,
    test_size=0.2, random_state=SEED
)

# ── DummyRegressor ─────────────────────────────────────────────────────
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_dummy))
r2_baseline   = r2_score(y_test, y_pred_dummy)

print('── Baseline (mean predictor) ──')
print(f'  RMSE : {rmse_baseline:.4f}  ← target must beat this')
print(f'  R²   : {r2_baseline:.4f}  ← should be ~0')
print(f'  std(y_train) = {y_train.std():.4f}  ← RMSE baseline ≈ this')
print()
print('Rule: RMSE_model / RMSE_baseline < 0.5 → good model')
print('      RMSE_model / RMSE_baseline > 0.9 → model barely helps')

---
## 2. Ridge Regression (L2)

### Идея

Линейная регрессия + штраф за большие коэффициенты.  
Коэффициенты **сжимаются к нулю**, но никогда не становятся ровно нулём.

```
Loss = Σ(y - ŷ)² + α × Σ(βⱼ²)
         OLS loss     L2 penalty
```

### Когда Ridge выигрывает

| Условие | Почему Ridge |
|---|---|
| Признаки сильно коррелируют (ρ > 0.7) | OLS неустойчив при мультиколлинеарности |
| Признаков много относительно n | Регуляризация снижает variance |
| Связь в данных линейная | Divergence (Spearman - Pearson) < 0.05 |
| Нужна интерпретируемость | Коэффициенты читаемы |

### Ключевой гиперпараметр: alpha

```
alpha = 0    → обычный OLS (нет регуляризации)
alpha мал    → слабый штраф, почти OLS
alpha велик  → сильный штраф, все β → 0
```

**Используй RidgeCV** — сам подбирает alpha через CV.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ── Ridge с фиксированным alpha ────────────────────────────────────────
ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),   # ОБЯЗАТЕЛЬНО: Ridge чувствителен к масштабу
    ('model',  Ridge(alpha=1.0, random_state=SEED))
])
ridge_pipe.fit(X_train, y_train)
y_pred_ridge = ridge_pipe.predict(X_test)

print('── Ridge (alpha=1.0) ──')
print(f'  RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_ridge)):.4f}')
print(f'  R²   : {r2_score(y_test, y_pred_ridge):.4f}')
print(f'  Spearman ρ: {spearmanr(y_test, y_pred_ridge).statistic:.4f}')
print()

# ── RidgeCV — авто-выбор alpha ─────────────────────────────────────────
# alphas — диапазон кандидатов (логарифмическая шкала лучше)
# cv     — количество фолдов
# scoring — метрика для выбора alpha
ridge_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RidgeCV(
        alphas=np.logspace(-3, 4, 50),  # от 0.001 до 10000
        cv=5,
        scoring='neg_mean_squared_error'
    ))
])
ridge_cv.fit(X_train, y_train)
best_alpha = ridge_cv.named_steps['model'].alpha_

print(f'── RidgeCV — best alpha = {best_alpha:.4f} ──')
print(f'  RMSE : {np.sqrt(mean_squared_error(y_test, ridge_cv.predict(X_test))):.4f}')
print(f'  R²   : {r2_score(y_test, ridge_cv.predict(X_test)):.4f}')

In [ ]:
# ── Коэффициенты Ridge: что влияет и в каком направлении ──────────────
# Важно: коэффициенты интерпретируемы только ПОСЛЕ StandardScaler
# (иначе единицы измерения разные)

feature_names = X_train.columns.tolist()
coefs = ridge_cv.named_steps['model'].coef_

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef':    coefs
}).sort_values('coef', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if c > 0 else 'coral' for c in coef_df['coef']]
ax.barh(coef_df['feature'], coef_df['coef'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Ridge Coefficients (alpha={best_alpha:.3f})\n'
             'Blue = positive effect, Red = negative effect')
ax.set_xlabel('Coefficient value (standardised)')
sns.despine()
plt.tight_layout()
plt.show()

print('\nTop features:')
print(coef_df.to_string(index=False))

---
## 3. Lasso (L1) & ElasticNet (L1 + L2)

### Lasso vs Ridge — главное отличие

```
Ridge (L2): Σβⱼ²    → коэффициенты → 0, но никогда = 0
Lasso (L1): Σ|βⱼ|   → часть коэффициентов = ровно 0 → автоматический feature selection
```

**Lasso делает sparse модель** — убивает ненужные признаки.

### Когда Lasso лучше Ridge

| Ситуация | Почему |
|---|---|
| Много признаков, подозреваешь что большинство бесполезны | Автовыбор признаков |
| Нужна интерпретируемость — минимум признаков | Читаемая модель |
| n << p (признаков больше наблюдений) | Lasso выберет только n значимых |

### ElasticNet = Lasso + Ridge

```
Loss = Σ(y-ŷ)² + α × [l1_ratio × Σ|βⱼ| + (1-l1_ratio) × Σβⱼ²]
```

- `l1_ratio = 1.0` → чистый Lasso  
- `l1_ratio = 0.0` → чистый Ridge  
- `l1_ratio = 0.5` → баланс (default)


In [ ]:
# ── Lasso ──────────────────────────────────────────────────────────────
lasso_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LassoCV(
        alphas=np.logspace(-4, 2, 50),
        cv=5,
        max_iter=10000,  # важно увеличить для сходимости
        random_state=SEED
    ))
])
lasso_cv.fit(X_train, y_train)
lasso_alpha = lasso_cv.named_steps['model'].alpha_
lasso_coefs = lasso_cv.named_steps['model'].coef_

n_zero  = (lasso_coefs == 0).sum()
n_kept  = (lasso_coefs != 0).sum()

print(f'── LassoCV (alpha={lasso_alpha:.4f}) ──')
print(f'  Features zeroed out : {n_zero} / {len(lasso_coefs)}')
print(f'  Features kept       : {n_kept}')
print(f'  RMSE : {np.sqrt(mean_squared_error(y_test, lasso_cv.predict(X_test))):.4f}')
print(f'  R²   : {r2_score(y_test, lasso_cv.predict(X_test)):.4f}')
print()

# Surviving features
surviving = pd.DataFrame({
    'feature': feature_names,
    'coef':    lasso_coefs
}).query('coef != 0').sort_values('coef', key=abs, ascending=False)
print('Surviving features (coef ≠ 0):')
print(surviving.to_string(index=False))

---
## 4. Logistic Regression (классификация)

### Идея

Линейная модель для классификации. Предсказывает **вероятность** принадлежности к классу.

```
P(y=1|x) = 1 / (1 + exp(-(β₀ + β₁x₁ + ... + βₚxₚ)))
           ↑
         sigmoid function → всегда в [0, 1]
```

**На DSI:** если таргет бинарный (0/1) или мультиклассовый — это твой linear baseline.

### Регуляризация в LogReg

```python
LogisticRegression(
    C=1.0,          # C = 1/alpha — ОБРАТНАЯ сила регуляризации
                    # C большой → слабая регуляризация (≈ нет)
                    # C малый  → сильная регуляризация
    penalty='l2',   # 'l2' (Ridge-like) | 'l1' (Lasso-like) | 'elasticnet'
    solver='lbfgs', # 'lbfgs' для l2, 'saga' для l1/elasticnet
    max_iter=1000,  # увеличь если не сходится
)
```

### Метрики для классификации

| Метрика | Когда | Формула |
|---|---|---|
| Accuracy | Балансированные классы | (TP+TN) / N |
| ROC-AUC | Любые классы, вероятности важны | Area under ROC curve |
| F1-score | Дисбаланс классов | 2 × P×R / (P+R) |
| Precision | Дорогие false positives | TP / (TP+FP) |
| Recall | Дорогие false negatives | TP / (TP+FN) |


In [ ]:
# Создаём бинарный таргет
y_binary = (y_raw > y_raw.median()).astype(int)
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_raw.select_dtypes(include=np.number), y_binary,
    test_size=0.2, random_state=SEED, stratify=y_binary  # stratify для баланса
)

# ── Logistic Regression ────────────────────────────────────────────────
logreg_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(
        C=1.0,
        penalty='l2',
        solver='lbfgs',
        max_iter=1000,
        random_state=SEED
    ))
])
logreg_pipe.fit(X_tr_b, y_tr_b)
y_pred_lr   = logreg_pipe.predict(X_te_b)
y_proba_lr  = logreg_pipe.predict_proba(X_te_b)[:, 1]

print('── Logistic Regression ──')
print(f'  Accuracy : {accuracy_score(y_te_b, y_pred_lr):.4f}')
print(f'  ROC-AUC  : {roc_auc_score(y_te_b, y_proba_lr):.4f}')
print()
print(classification_report(y_te_b, y_pred_lr))
print()
print('Правило: ROC-AUC = 0.5 → случайно, = 1.0 → идеально')
print('         ROC-AUC > 0.7 → работающая модель')
print('         ROC-AUC > 0.9 → проверь на leakage!')

---
## 5. LightGBM

### Идея — Gradient Boosting

500 деревьев последовательно. Каждое следующее исправляет ошибки предыдущего.

```
Дерево 1: предсказывает грубо
    ↓ остатки (residuals)
Дерево 2: предсказывает остатки
    ↓ новые остатки
...
Финал = Σ (learning_rate × дерево_t)
```

### Leaf-wise vs Level-wise (отличие от XGBoost)

```
LightGBM (leaf-wise):        XGBoost (level-wise):
идёт туда где max ошибка     растёт равномерно по уровням
→ быстрее, точнее            → консервативнее
→ риск overfit на малых n    → стабильнее
```

### Гиперпараметры — полный разбор

| Параметр | Default | Влияние | Когда менять |
|---|---|---|---|
| `n_estimators` | 100 | Кол-во деревьев | Больше = точнее, но медленнее |
| `learning_rate` | 0.1 | Шаг обучения | Меньше + больше n_estimators = лучше |
| `num_leaves` | 31 | Сложность дерева | Уменьши при overfit |
| `min_child_samples` | 20 | Мин. obs в листе | Увеличь при overfit |
| `subsample` | 1.0 | Доля строк/дерево | 0.7-0.9 снижает overfit |
| `colsample_bytree` | 1.0 | Доля признаков | 0.7-0.9 снижает overfit |
| `reg_alpha` | 0.0 | L1 регуляризация | Если нужна sparse модель |
| `reg_lambda` | 0.0 | L2 регуляризация | Добавить при overfit |


In [ ]:
# ── LightGBM Regressor ─────────────────────────────────────────────────
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,        # меньше → плавнее обучение
    num_leaves=64,             # complexity: 2^(max_depth)
    min_child_samples=20,      # min obs per leaf
    subsample=0.8,             # 80% строк на каждое дерево
    colsample_bytree=0.8,      # 80% признаков на каждое дерево
    reg_alpha=0.0,             # L1
    reg_lambda=0.0,            # L2
    importance_type='gain',    # gain > split для важности
    random_state=SEED,
    verbose=-1
)

lgb_model.fit(X_train, y_train)
y_pred_lgb = lgb_model.predict(X_test)

print('── LightGBM ──')
print(f'  RMSE_train : {np.sqrt(mean_squared_error(y_train, lgb_model.predict(X_train))):.4f}')
print(f'  RMSE_test  : {np.sqrt(mean_squared_error(y_test, y_pred_lgb)):.4f}')
print(f'  R²         : {r2_score(y_test, y_pred_lgb):.4f}')
print(f'  Spearman ρ : {spearmanr(y_test, y_pred_lgb).statistic:.4f}')
print(f'  Overfit gap: {np.sqrt(mean_squared_error(y_test, y_pred_lgb)) - np.sqrt(mean_squared_error(y_train, lgb_model.predict(X_train))):+.4f}')

In [ ]:
# ── Feature Importance (gain) ──────────────────────────────────────────
# gain = суммарное снижение ошибки от этого признака по всем деревьям
# Нормируем в % — легче интерпретировать

fi = pd.DataFrame({
    'feature':    feature_names,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False)
fi['pct'] = fi['importance'] / fi['importance'].sum() * 100

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=fi, x='pct', y='feature', color='steelblue', ax=ax)
ax.set_xlabel('% of total gain')
ax.set_title('LightGBM Feature Importance (Gain)\n'
             'One feature > 60% → check for leakage')
sns.despine()
plt.tight_layout()
plt.show()

print(fi[['feature','pct']].to_string(index=False))

In [ ]:
# ── Диагностика overfitting ────────────────────────────────────────────
# Learning curve: как RMSE меняется с ростом train size
# Если train и val кривые сходятся → OK
# Если большой gap → overfit

train_sizes, train_scores, val_scores = learning_curve(
    lgb_model, X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

train_rmse = np.sqrt(-train_scores.mean(axis=1))
val_rmse   = np.sqrt(-val_scores.mean(axis=1))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_sizes, train_rmse, 'o-', color='steelblue', label='Train RMSE')
ax.plot(train_sizes, val_rmse,   'o-', color='coral',     label='Val RMSE')
ax.fill_between(train_sizes,
    np.sqrt(-train_scores.mean(axis=1)) - np.sqrt(-train_scores).std(axis=1),
    np.sqrt(-train_scores.mean(axis=1)) + np.sqrt(-train_scores).std(axis=1),
    alpha=0.15, color='steelblue')
ax.set_xlabel('Training set size')
ax.set_ylabel('RMSE')
ax.set_title('Learning Curve — LightGBM\n'
             'Large gap = overfit | Both curves high = underfit | Converging = OK')
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

---
## 6. XGBoost

### Отличие от LightGBM

```
XGBoost: level-wise (depth-first)  →  медленнее, но стабильнее
LightGBM: leaf-wise                →  быстрее, но агрессивнее

XGBoost лучше на:  маленьких датасетах, шумных данных
LightGBM лучше на: больших датасетах, многих признаках
```

### Гиперпараметры XGBoost

| Параметр | Аналог LightGBM | Влияние |
|---|---|---|
| `n_estimators` | `n_estimators` | Кол-во деревьев |
| `learning_rate` | `learning_rate` | Шаг обучения |
| `max_depth` | `num_leaves` | Глубина дерева (3-6) |
| `min_child_weight` | `min_child_samples` | Мин. сумма весов в листе |
| `subsample` | `subsample` | Доля строк |
| `colsample_bytree` | `colsample_bytree` | Доля признаков |
| `reg_alpha` | `reg_alpha` | L1 |
| `reg_lambda` | `reg_lambda` | L2 (default=1, уже есть!) |

### Лечение overfit в XGBoost

```python
# Консервативная конфигурация для малых датасетов:
xgb.XGBRegressor(
    max_depth=3,            # было 6 → уменьши
    min_child_weight=10,    # было 1 → увеличь
    subsample=0.6,          # было 1 → уменьши
    reg_alpha=1.0,          # добавь L1
    reg_lambda=5.0,         # усиль L2
)
```


In [ ]:
# ── XGBoost ────────────────────────────────────────────────────────────
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=5,    # min sum of weights in a leaf
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,         # L1
    reg_lambda=1.0,        # L2 — default в XGBoost уже 1!
    random_state=SEED,
    verbosity=0
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

rmse_train_xgb = np.sqrt(mean_squared_error(y_train, xgb_model.predict(X_train)))
rmse_test_xgb  = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

print('── XGBoost ──')
print(f'  RMSE_train : {rmse_train_xgb:.4f}')
print(f'  RMSE_test  : {rmse_test_xgb:.4f}')
print(f'  R²         : {r2_score(y_test, y_pred_xgb):.4f}')
print(f'  Spearman ρ : {spearmanr(y_test, y_pred_xgb).statistic:.4f}')
print(f'  Overfit gap: {rmse_test_xgb - rmse_train_xgb:+.4f}')

---
## 7. Сравнение моделей — когда что выбрать

```
Датасет получен → смотришь на структуру
        │
        ├── Divergence (Spearman - Pearson) < 0.05?
        │   → связь ЛИНЕЙНАЯ → Ridge / Lasso
        │
        ├── Divergence > 0.05?
        │   → связь НЕЛИНЕЙНАЯ → LightGBM / XGBoost
        │
        ├── n < 1000 строк?
        │   → XGBoost консервативнее, меньше overfit
        │
        ├── Много пропусков / категориальных?
        │   → LightGBM (встроенная поддержка)
        │
        ├── Нужна интерпретируемость?
        │   → Ridge (коэффициенты) или Lasso (sparse)
        │
        └── Бинарный таргет?
            → Logistic Regression как baseline, LightGBM как main
```


In [ ]:
# ── Полное сравнение всех моделей ─────────────────────────────────────

def full_eval(model, X_tr, y_tr, X_te, y_te, name):
    model.fit(X_tr, y_tr)
    p_tr = model.predict(X_tr)
    p_te = model.predict(X_te)
    return {
        'model':       name,
        'RMSE_train':  np.sqrt(mean_squared_error(y_tr, p_tr)),
        'RMSE_test':   np.sqrt(mean_squared_error(y_te, p_te)),
        'MAE_test':    mean_absolute_error(y_te, p_te),
        'R2_test':     r2_score(y_te, p_te),
        'Spearman':    spearmanr(y_te, p_te).statistic,
        'overfit_gap': np.sqrt(mean_squared_error(y_te, p_te)) -
                       np.sqrt(mean_squared_error(y_tr, p_tr)),
    }

results = [
    full_eval(DummyRegressor(strategy='mean'),          X_train, y_train, X_test, y_test, 'Baseline'),
    full_eval(ridge_cv,                                  X_train, y_train, X_test, y_test, 'Ridge CV'),
    full_eval(lasso_cv,                                  X_train, y_train, X_test, y_test, 'Lasso CV'),
    full_eval(lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                                 num_leaves=64, random_state=SEED, verbose=-1),
              X_train, y_train, X_test, y_test, 'LightGBM'),
    full_eval(xgb.XGBRegressor(n_estimators=500, learning_rate=0.05,
                                max_depth=5, random_state=SEED, verbosity=0),
              X_train, y_train, X_test, y_test, 'XGBoost'),
]

res_df = pd.DataFrame(results).set_index('model')
print('Full Model Comparison:')
print(res_df.round(4).to_string())

# ── Визуализация ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(res_df))
axes[0].bar(x - 0.2, res_df['RMSE_train'], 0.35, label='Train', color='steelblue', alpha=0.85)
axes[0].bar(x + 0.2, res_df['RMSE_test'],  0.35, label='Test',  color='coral',     alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(res_df.index, rotation=15)
axes[0].set_ylabel('RMSE')
axes[0].set_title('Train vs Test RMSE\nLarge gap = overfit')
axes[0].legend()

axes[1].bar(res_df.index, res_df['Spearman'], color='steelblue', alpha=0.85)
axes[1].set_ylabel('Spearman ρ')
axes[1].set_title('Rank Correlation with Actuals\n(Spearman ρ) — key quant metric')
axes[1].tick_params(axis='x', rotation=15)

sns.despine()
plt.tight_layout()
plt.show()

---
## 8. Cross-Validation стратегии

### Зачем CV

Один train/test split — ненадёжен. Метрика может случайно быть хорошей или плохой.  
CV даёт **среднее ± std** по нескольким сплитам → стабильная оценка.

### Три стратегии

| Стратегия | Когда | Особенность |
|---|---|---|
| **K-Fold** | Данные i.i.d. (независимые) | Стандарт |
| **Stratified K-Fold** | Классификация, дисбаланс классов | Сохраняет пропорции классов |
| **TimeSeriesSplit** | Временные ряды | Train всегда раньше Val по времени |

### Важное правило — data leakage в CV

```
НЕПРАВИЛЬНО: fit StandardScaler на всех данных, потом CV
ПРАВИЛЬНО:   Pipeline = Scaler + Model → CV на Pipeline
             Scaler fitится только на train fold
```


In [ ]:
# ── K-Fold CV ──────────────────────────────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

# ПРАВИЛЬНО: CV на Pipeline — scaler не видит val fold
ridge_pipe_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Ridge(alpha=1.0))
])

scores = cross_val_score(
    ridge_pipe_cv, X_train, y_train,
    cv=kf,
    scoring='neg_mean_squared_error'
)
rmse_cv = np.sqrt(-scores)

print('── 5-Fold CV: Ridge ──')
print(f'  RMSE per fold: {[f"{x:.3f}" for x in rmse_cv]}')
print(f'  Mean RMSE : {rmse_cv.mean():.4f}')
print(f'  Std  RMSE : {rmse_cv.std():.4f}')
print(f'  CV stable? std/mean = {rmse_cv.std()/rmse_cv.mean():.3f}  '
      f'(< 0.05 = very stable, > 0.15 = unstable)')

In [ ]:
# ── Walk-Forward (TimeSeriesSplit) ─────────────────────────────────────
# Expanding window: каждый следующий fold добавляет данные в train
# НИКОГДА не используй K-Fold для временных рядов — это look-ahead bias

tscv = TimeSeriesSplit(n_splits=5)

wf_scores = []
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
    X_tr_f, X_val_f = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr_f, y_val_f = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    m = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05,
                           random_state=SEED, verbose=-1)
    m.fit(X_tr_f, y_tr_f)
    pred = m.predict(X_val_f)
    rmse = np.sqrt(mean_squared_error(y_val_f, pred))
    rho  = spearmanr(y_val_f, pred).statistic
    wf_scores.append({'fold': fold+1, 'n_train': len(tr_idx),
                      'RMSE': rmse, 'Spearman': rho})
    print(f'  Fold {fold+1}: train={len(tr_idx):5,}  RMSE={rmse:.4f}  ρ={rho:.4f}')

wf_df = pd.DataFrame(wf_scores)
print(f'\n  Mean RMSE     : {wf_df["RMSE"].mean():.4f} ± {wf_df["RMSE"].std():.4f}')
print(f'  Mean Spearman : {wf_df["Spearman"].mean():.4f} ± {wf_df["Spearman"].std():.4f}')
print()
print('Стабильный: std/mean < 0.05')
print(f'Твой: std/mean = {wf_df["RMSE"].std()/wf_df["RMSE"].mean():.3f}')

---
## 9. Overfitting — диагностика и лечение

### Три состояния модели

```
                    RMSE_train   RMSE_test   gap
Underfitting:        высокий      высокий     мал
Good fit:            низкий       низкий      мал
Overfitting:         очень низкий  высокий   БОЛЬШОЙ
```

### Overfitting gap — пороги

```
gap < 0.05 × RMSE_test   → отлично, нет overfit
gap < 0.15 × RMSE_test   → приемлемо
gap > 0.15 × RMSE_test   → overfit, нужно лечить
gap > 0.5  × RMSE_test   → серьёзный overfit
```

### Рецепт лечения для каждой модели

| Модель | Лечение |
|---|---|
| Ridge | Увеличь alpha (RidgeCV автоматом) |
| LightGBM | ↓ num_leaves, ↑ min_child_samples, ↓ subsample |
| XGBoost | ↓ max_depth, ↑ min_child_weight, ↑ reg_lambda |
| Все | Больше данных, feature engineering, dropout |


In [ ]:
# ── Симулируем overfit: намеренно мало данных ──────────────────────────
X_small, _, y_small, _ = train_test_split(X_train, y_train,
                                             train_size=0.1, random_state=SEED)

overfit_model = lgb.LGBMRegressor(
    n_estimators=1000,
    num_leaves=128,       # слишком сложно для малых данных
    min_child_samples=1,  # один obs в листе = выучивание шума
    random_state=SEED, verbose=-1
)
overfit_model.fit(X_small, y_small)

rmse_tr = np.sqrt(mean_squared_error(y_small, overfit_model.predict(X_small)))
rmse_te = np.sqrt(mean_squared_error(y_test,  overfit_model.predict(X_test)))

print('── Overfit model (deliberately bad) ──')
print(f'  RMSE_train : {rmse_tr:.4f}  ← near zero')
print(f'  RMSE_test  : {rmse_te:.4f}  ← much worse')
print(f'  Gap        : {rmse_te - rmse_tr:+.4f}')
print(f'  Gap / Test : {(rmse_te - rmse_tr) / rmse_te:.2%}  ← > 50% = severe overfit')
print()

# ── Лечение: регуляризация ──────────────────────────────────────────────
fixed_model = lgb.LGBMRegressor(
    n_estimators=200,
    num_leaves=16,        # было 128 → уменьшили
    min_child_samples=20, # было 1  → увеличили
    subsample=0.6,        # добавили стохастику
    colsample_bytree=0.6,
    reg_lambda=1.0,       # добавили L2
    random_state=SEED, verbose=-1
)
fixed_model.fit(X_small, y_small)

rmse_tr2 = np.sqrt(mean_squared_error(y_small, fixed_model.predict(X_small)))
rmse_te2 = np.sqrt(mean_squared_error(y_test,  fixed_model.predict(X_test)))

print('── Fixed model ──')
print(f'  RMSE_train : {rmse_tr2:.4f}')
print(f'  RMSE_test  : {rmse_te2:.4f}')
print(f'  Gap        : {rmse_te2 - rmse_tr2:+.4f}  ← much smaller')
print(f'  Gap / Test : {(rmse_te2 - rmse_tr2) / rmse_te2:.2%}')

---
## 10. 🚀 Copy-Paste шаблон — полный pipeline

Вставь в ноутбук, замени `TARGET` и `DATA_PATH`.  
Работает для любого табличного датасета.


### Шаг 1 — Загрузка и preprocessing

In [ ]:
# ═══════════════════════════════════════════════════════════
# ШАБЛОН: ПОЛНЫЙ PIPELINE ДЛЯ DSI
# Замени TARGET, DATA_PATH, и типы задачи
# ═══════════════════════════════════════════════════════════

TARGET    = 'YOUR_TARGET'    # ← ЗАМЕНИТЬ
DATA_PATH = 'data.csv'       # ← ЗАМЕНИТЬ
TASK      = 'regression'     # 'regression' или 'classification'
SEED      = 42
K_FOLDS   = 5

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')

# Убираем leakage колонки (вручную после EDA)
LEAKAGE_COLS = []  # ← ЗАМЕНИТЬ после EDA
df_clean = df.drop(columns=[c for c in LEAKAGE_COLS if c in df.columns])

# Features и target
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED,
    stratify=y if TASK == 'classification' else None
)

# Preprocessing pipeline
num_features = X.select_dtypes(include=np.number).columns.tolist()
cat_features = X.select_dtypes(include='object').columns.tolist()

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features)
], remainder='drop')

X_train_p = preprocessor.fit_transform(X_train)  # fit only on train!
X_test_p  = preprocessor.transform(X_test)
print(f'Processed: {X_train_p.shape}')

### Шаг 2 — Evaluation helper

In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te, name='Model', k=K_FOLDS):
    """Universal evaluator: CV + hold-out + all metrics."""
    # K-Fold CV
    kf = KFold(n_splits=k, shuffle=True, random_state=SEED)
    scoring = 'neg_mean_squared_error' if TASK == 'regression' else 'roc_auc'
    cv_raw = cross_val_score(model, X_tr, y_tr, cv=kf, scoring=scoring)
    cv_mean = (np.sqrt(-cv_raw) if TASK == 'regression' else cv_raw).mean()
    cv_std  = (np.sqrt(-cv_raw) if TASK == 'regression' else cv_raw).std()

    model.fit(X_tr, y_tr)
    p_tr = model.predict(X_tr)
    p_te = model.predict(X_te)

    if TASK == 'regression':
        m = {
            'model':       name,
            'CV_mean':     cv_mean,
            'CV_std':      cv_std,
            'RMSE_train':  np.sqrt(mean_squared_error(y_tr, p_tr)),
            'RMSE_test':   np.sqrt(mean_squared_error(y_te, p_te)),
            'MAE_test':    mean_absolute_error(y_te, p_te),
            'R2_test':     r2_score(y_te, p_te),
            'Spearman':    spearmanr(y_te, p_te).statistic,
        }
        m['overfit_gap'] = m['RMSE_test'] - m['RMSE_train']
    else:
        p_proba = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else p_te
        m = {
            'model':      name,
            'CV_mean':    cv_mean, 'CV_std': cv_std,
            'Accuracy':   accuracy_score(y_te, p_te),
            'ROC_AUC':    roc_auc_score(y_te, p_proba),
        }

    print(f'\n── {name} ──')
    for k_,v_ in m.items():
        if k_ != 'model':
            print(f'  {k_:15s}: {v_:.4f}')
    return m

### Шаг 3 — Train all models

In [ ]:
results = []

# 1. Baseline
baseline = DummyRegressor(strategy='mean') if TASK == 'regression' \
           else DummyClassifier(strategy='most_frequent')
results.append(evaluate(baseline, X_train_p, y_train, X_test_p, y_test, 'Baseline'))

# 2. Ridge (only for regression)
if TASK == 'regression':
    ridge = RidgeCV(alphas=np.logspace(-3, 4, 50), cv=5)
    results.append(evaluate(ridge, X_train_p, y_train, X_test_p, y_test, 'Ridge CV'))

# 3. Logistic Regression (only for classification)
if TASK == 'classification':
    logreg = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    results.append(evaluate(logreg, X_train_p, y_train, X_test_p, y_test, 'LogReg'))

# 4. LightGBM
lgb_params = dict(n_estimators=500, learning_rate=0.05, num_leaves=64,
                  min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                  importance_type='gain', random_state=SEED, verbose=-1)
lgb_m = lgb.LGBMRegressor(**lgb_params) if TASK == 'regression' \
       else lgb.LGBMClassifier(**lgb_params)
results.append(evaluate(lgb_m, X_train_p, y_train, X_test_p, y_test, 'LightGBM'))

# 5. XGBoost
xgb_params = dict(n_estimators=500, learning_rate=0.05, max_depth=5,
                  subsample=0.8, colsample_bytree=0.8,
                  reg_alpha=0.1, reg_lambda=1.0,
                  random_state=SEED, verbosity=0)
xgb_m = xgb.XGBRegressor(**xgb_params) if TASK == 'regression' \
       else xgb.XGBClassifier(**xgb_params)
results.append(evaluate(xgb_m, X_train_p, y_train, X_test_p, y_test, 'XGBoost'))

# Summary table
res_df = pd.DataFrame(results).set_index('model')
print('\n═══ FULL COMPARISON ═══')
print(res_df.round(4).to_string())

### Шаг 4 — Residuals & diagnostics

In [ ]:
# ── Получаем лучшую модель ─────────────────────────────────────────────
best_model = lgb_m  # ← замени на победителя из таблицы
y_pred_best = best_model.predict(X_test_p)
residuals   = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Residuals vs Predicted
sns.scatterplot(x=y_pred_best, y=residuals, alpha=0.4, s=15,
                color='steelblue', ax=axes[0])
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set(xlabel='Predicted', ylabel='Residual',
            title='Residuals vs Predicted\nFan = heteroskedasticity | Curve = non-linearity')

# 2. Distribution
sns.histplot(residuals, bins=40, kde=True, color='steelblue',
             edgecolor='white', ax=axes[1])
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set(title=f'Residuals  mean={residuals.mean():.3f}  std={residuals.std():.3f}')

# 3. QQ
from scipy import stats as scipy_stats
(osm, osr), (slope, intercept, r) = scipy_stats.probplot(residuals, dist='norm')
axes[2].plot(osm, osr, 'o', alpha=0.4, s=4, color='steelblue')
axes[2].plot(osm, slope*np.array(osm)+intercept, 'r--', lw=1.5)
axes[2].set(title=f'Normal Q-Q  R={r:.3f}', xlabel='Theoretical quantiles')

sns.despine()
plt.tight_layout()
plt.show()

# Error stratification
err_df = pd.DataFrame({'actual': y_test.values, 'pred': y_pred_best,
                        'abs_err': np.abs(residuals)})
err_df['bin'] = pd.cut(err_df['actual'], bins=5)
strat = err_df.groupby('bin', observed=True).agg(
    n=('actual','count'),
    RMSE=('abs_err', lambda x: np.sqrt((x**2).mean()))
).round(4)
print('RMSE by actual value range:')
print(strat.to_string())